# Capstone Project " Bayesian Optimisation Challenge
This notebook is used to explore the initial data for all 8 black-box functions and plan query strategies.

In [ ]:
import numpy as np

base_path = r'C:/Users/Owner/Documents/Terraform2/capstone/Initial_data_points_starter/initial_data'

functions = {}

for i in range(1, 9):
    inputs = np.load(f'{base_path}/function_{i}/initial_inputs.npy')
    outputs = np.load(f'{base_path}/function_{i}/initial_outputs.npy')
    functions[i] = {'inputs': inputs, 'outputs': outputs}
    print(f'--- Function {i} ---')
    print(f'Inputs shape:  {inputs.shape}')
    print(f'Outputs shape: {outputs.shape}')
    print(f'Input values:\n{inputs}')
    print(f'Output values: {outputs}')
    print(f'Best output so far: {np.max(outputs):.6f} at index {np.argmax(outputs)}')
    print()

## Best Input Per Function\nFor each function, find the input that produced the highest output.

In [ ]:
for i in range(1, 9):
    inputs = functions[i]['inputs']
    outputs = functions[i]['outputs']
    best_index = np.argmax(outputs)
    best_input = inputs[best_index]
    best_output = outputs[best_index]
    print(f'--- Function {i} ---')
    print(f'Best input:  {best_input}')
    print(f'Best output: {best_output:.6f}')
    print()

In [ ]:
# Nudge amount " adjust this value to control how much we shift the best input
nudge = 0.05

for i in range(1, 9):
    inputs = functions[i]['inputs']
    outputs = functions[i]['outputs']
    best_index = np.argmax(outputs)
    best_input = inputs[best_index]
    
    # Add nudge to each dimension
    suggested_query = best_input + nudge
    
    # Clip to valid range [0, 1] in case nudge pushes values out of bounds
    suggested_query = np.clip(suggested_query, 0, 1)
    
    print(f'--- Function {i} ---')
    print(f'Best input:       {best_input}')
    print(f'Suggested query:  {suggested_query}')
    print()

## Round 2 " Bayesian Optimisation with GP and UCB
For each function, we update the data with Round 1 results, fit a GP, and use UCB to suggest the next query point.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Round 1 results " inputs submitted and outputs received
round_1 = {
    1: {'input': [0.781024, 0.783000],                                                          'output': 3.60011e-33},
    2: {'input': [0.752637, 0.975656],                                                          'output': 0.3504834},
    3: {'input': [0.542581, 0.661593, 0.390176],                                                'output': -0.0234504},
    4: {'input': [0.627766, 0.478772, 0.475826, 0.299007],                                     'output': -6.5008454},
    5: {'input': [0.274189, 0.896480, 0.929484, 0.928516],                                     'output': 1961.0955629},
    6: {'input': [0.778186, 0.204693, 0.782552, 0.743997, 0.106401],                           'output': -0.6397542},
    7: {'input': [0.107896, 0.541672, 0.297422, 0.268118, 0.470428, 0.780970],                 'output': 1.0204860},
    8: {'input': [0.106447, 0.115956, 0.072929, 0.088786, 0.453935, 0.851055, 0.538307, 0.943085], 'output': 9.5614453},
}

# UCB beta " controls exploration vs exploitation (higher = more exploration)
beta = 2.0

# Number of random candidate points to evaluate the acquisition function over
n_candidates = 10000

for i in range(1, 9):
    # Combine initial data with Round 1 result
    X = np.vstack([functions[i]['inputs'], round_1[i]['input']])
    y = np.append(functions[i]['outputs'], round_1[i]['output'])

    # Normalise outputs for GP stability
    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = (y - y_mean) / y_std

    # Fit GP
    kernel = C(1.0) * RBF(length_scale=1.0)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=False)
    gp.fit(X, y_norm)

    # Generate random candidates across [0, 1] for each dimension
    n_dims = X.shape[1]
    candidates = np.random.uniform(0, 1, size=(n_candidates, n_dims))

    # UCB acquisition function
    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + beta * sigma

    # Select best candidate
    best_candidate = candidates[np.argmax(ucb)]

    print(f'--- Function {i} ---')
    print(f'Suggested query: {best_candidate}')
    print(f'Formatted:       {"-".join(f"{v:.6f}" for v in best_candidate)}')
    print()

## Round 3 " Ensemble GP with Latin Hypercube Sampling
Changes from Round 2:
- **Ensemble of 3 kernels** (RBF, Matern nu=2.5, Matern nu=1.5) " UCB scores averaged across all three, reducing risk of a single kernel being a poor fit
- **Latin Hypercube Sampling** for candidates " guarantees even coverage across all dimensions, avoiding gaps from pure random sampling
- **Beta reduced to 1.0** " favour exploitation; Round 2 showed over-exploration backfired badly on Function 5
- **Widened length_scale_bounds** (1e-6 to 1e3) " fixes ConvergenceWarning from Round 2

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel as C
from scipy.stats.qmc import LatinHypercube

# Round 1 results
round_1 = {
    1: {'input': [0.781024, 0.783000],                                                              'output': 3.60011e-33},
    2: {'input': [0.752637, 0.975656],                                                              'output': 0.3504834},
    3: {'input': [0.542581, 0.661593, 0.390176],                                                    'output': -0.0234504},
    4: {'input': [0.627766, 0.478772, 0.475826, 0.299007],                                         'output': -6.5008454},
    5: {'input': [0.274189, 0.896480, 0.929484, 0.928516],                                         'output': 1961.0955629},
    6: {'input': [0.778186, 0.204693, 0.782552, 0.743997, 0.106401],                               'output': -0.6397542},
    7: {'input': [0.107896, 0.541672, 0.297422, 0.268118, 0.470428, 0.780970],                     'output': 1.0204860},
    8: {'input': [0.106447, 0.115956, 0.072929, 0.088786, 0.453935, 0.851055, 0.538307, 0.943085], 'output': 9.5614453},
}

# Round 2 results
round_2 = {
    1: {'input': [0.722566, 0.786336],                                                              'output': -5.8376561e-24},
    2: {'input': [0.338518, 0.489826],                                                              'output': 0.0381634},
    3: {'input': [0.353706, 0.737070, 0.756926],                                                    'output': -0.1039563},
    4: {'input': [0.548017, 0.538375, 0.021034, 0.258912],                                         'output': -10.9738436},
    5: {'input': [0.225843, 0.909370, 0.503192, 0.299769],                                         'output': 58.7131993},
    6: {'input': [0.386227, 0.370287, 0.535411, 0.895749, 0.315540],                               'output': -0.4914677},
    7: {'input': [0.089709, 0.438459, 0.180432, 0.183812, 0.354751, 0.831516],                     'output': 1.3918175},
    8: {'input': [0.443569, 0.599844, 0.033920, 0.013473, 0.948748, 0.067015, 0.092241, 0.036099], 'output': 9.2616659},
}

# Ensemble of kernels " hedges against not knowing the true function shape
kernels = [
    C(1.0) * RBF(length_scale=1.0,    length_scale_bounds=(1e-6, 1e3)),
    C(1.0) * Matern(length_scale=1.0, length_scale_bounds=(1e-6, 1e3), nu=2.5),
    C(1.0) * Matern(length_scale=1.0, length_scale_bounds=(1e-6, 1e3), nu=1.5),
]

# Beta = 1.0 " favour exploitation over exploration
beta = 1.0

# Number of candidates
n_candidates = 10000

for i in range(1, 9):
    # Stack initial data + Round 1 + Round 2 (13 points per function)
    X = np.vstack([
        functions[i]['inputs'],
        round_1[i]['input'],
        round_2[i]['input']
    ])
    y = np.concatenate([
        functions[i]['outputs'],
        [round_1[i]['output']],
        [round_2[i]['output']]
    ])

    # Normalise outputs for GP stability
    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = (y - y_mean) / y_std

    # Latin Hypercube Sampling " even coverage across all dimensions
    n_dims = X.shape[1]
    sampler = LatinHypercube(d=n_dims)
    candidates = sampler.random(n=n_candidates)

    # Fit each kernel and accumulate UCB scores
    ensemble_ucb = np.zeros(n_candidates)
    for kernel in kernels:
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=False)
        gp.fit(X, y_norm)
        mu, sigma = gp.predict(candidates, return_std=True)
        ensemble_ucb += mu + beta * sigma

    # Average UCB across all kernels and select best candidate
    ensemble_ucb /= len(kernels)
    best_candidate = candidates[np.argmax(ensemble_ucb)]

    print(f'--- Function {i} ---')
    print(f'Suggested query: {best_candidate}')
    print(f'Formatted:       {"-".join(f"{v:.6f}" for v in best_candidate)}')
    print()

## Round 4 " Hybrid Strategy
Tailored method per function based on accumulated evidence:

- **Function 1** " Maximin exploration: all outputs ~0, no surrogate can learn here; place query furthest from all existing data
- **Functions 2, 3, 4, 7, 8** " GP ensemble + LHS + beta=1.0: all improved in Round 3, no reason to change what is working
- **Function 5** " Neural network surrogate + gradient ascent from Round 1 best `[0.274, 0.896, 0.929, 0.929]`: we know the good region, need precise local refinement to push past 1961 " gradient ascent follows the surrogate's slope toward the nearby peak
- **Function 6** " Neural network surrogate + gradient ascent from Round 2 best `[0.386, 0.370, 0.535, 0.896, 0.316]`: GP has drifted away from the only good result two rounds running " warm-starting from that point exploits what we know works

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel as C
from scipy.stats.qmc import LatinHypercube

# Round 1 results
round_1 = {
    1: {'input': [0.781024, 0.783000],                                                              'output': 3.60011e-33},
    2: {'input': [0.752637, 0.975656],                                                              'output': 0.3504834},
    3: {'input': [0.542581, 0.661593, 0.390176],                                                    'output': -0.0234504},
    4: {'input': [0.627766, 0.478772, 0.475826, 0.299007],                                         'output': -6.5008454},
    5: {'input': [0.274189, 0.896480, 0.929484, 0.928516],                                         'output': 1961.0955629},
    6: {'input': [0.778186, 0.204693, 0.782552, 0.743997, 0.106401],                               'output': -0.6397542},
    7: {'input': [0.107896, 0.541672, 0.297422, 0.268118, 0.470428, 0.780970],                     'output': 1.0204860},
    8: {'input': [0.106447, 0.115956, 0.072929, 0.088786, 0.453935, 0.851055, 0.538307, 0.943085], 'output': 9.5614453},
}

# Round 2 results
round_2 = {
    1: {'input': [0.722566, 0.786336],                                                              'output': -5.8376561e-24},
    2: {'input': [0.338518, 0.489826],                                                              'output': 0.0381634},
    3: {'input': [0.353706, 0.737070, 0.756926],                                                    'output': -0.1039563},
    4: {'input': [0.548017, 0.538375, 0.021034, 0.258912],                                         'output': -10.9738436},
    5: {'input': [0.225843, 0.909370, 0.503192, 0.299769],                                         'output': 58.7131993},
    6: {'input': [0.386227, 0.370287, 0.535411, 0.895749, 0.315540],                               'output': -0.4914677},
    7: {'input': [0.089709, 0.438459, 0.180432, 0.183812, 0.354751, 0.831516],                     'output': 1.3918175},
    8: {'input': [0.443569, 0.599844, 0.033920, 0.013473, 0.948748, 0.067015, 0.092241, 0.036099], 'output': 9.2616659},
}

# Round 3 results
round_3 = {
    1: {'input': [0.787451, 0.713163],                                                              'output': -5.8000091e-23},
    2: {'input': [0.666136, 0.924650],                                                              'output': 0.4682318},
    3: {'input': [0.432014, 0.423353, 0.507804],                                                    'output': -0.0177134},
    4: {'input': [0.469321, 0.416892, 0.307101, 0.426050],                                         'output': -1.3139071},
    5: {'input': [0.264682, 0.835123, 0.958851, 0.940666],                                         'output': 1909.4865637},
    6: {'input': [0.263435, 0.869356, 0.592685, 0.722761, 0.544815],                               'output': -1.2691502},
    7: {'input': [0.085295, 0.293495, 0.173384, 0.126868, 0.393363, 0.660213],                     'output': 1.7092281},
    8: {'input': [0.104083, 0.054438, 0.407037, 0.006596, 0.900508, 0.470715, 0.078870, 0.916641], 'output': 9.6947417},
}

# GP ensemble kernels (same as Round 3 " validated)
gp_kernels = [
    C(1.0) * RBF(length_scale=1.0,    length_scale_bounds=(1e-6, 1e3)),
    C(1.0) * Matern(length_scale=1.0, length_scale_bounds=(1e-6, 1e3), nu=2.5),
    C(1.0) * Matern(length_scale=1.0, length_scale_bounds=(1e-6, 1e3), nu=1.5),
]
beta = 1.0
n_candidates = 10000

def gp_ensemble_query(X, y, n_dims):
    """GP ensemble + LHS " same approach as Round 3."""
    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = (y - y_mean) / y_std
    sampler = LatinHypercube(d=n_dims)
    candidates = sampler.random(n=n_candidates)
    ensemble_ucb = np.zeros(n_candidates)
    for kernel in gp_kernels:
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=False)
        gp.fit(X, y_norm)
        mu, sigma = gp.predict(candidates, return_std=True)
        ensemble_ucb += mu + beta * sigma
    ensemble_ucb /= len(gp_kernels)
    return candidates[np.argmax(ensemble_ucb)]

def nn_gradient_ascent_query(X, y, warm_start, radius, n_random_starts=9, lr=0.05, n_steps=500):
    """Neural network surrogate + gradient ascent constrained to a local ball around warm_start.
    radius controls how far the search can stray from the warm start in each dimension."""
    n_dims = X.shape[1]
    warm_start = np.array(warm_start, dtype=np.float32)

    # Per-dimension bounds: warm_start +/- radius, clipped to [0, 1]
    lo = np.clip(warm_start - radius, 0.0, 1.0)
    hi = np.clip(warm_start + radius, 0.0, 1.0)

    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = ((y - y_mean) / y_std).reshape(-1, 1).astype(np.float32)
    X_f = X.astype(np.float32)

    # Small MLP with L2 regularisation " prevents overfitting on 14 points
    reg = tf.keras.regularizers.L2(0.01)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(8, activation='relu', kernel_regularizer=reg, input_shape=(n_dims,)),
        tf.keras.layers.Dense(8, activation='relu', kernel_regularizer=reg),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse')
    model.fit(X_f, y_norm, epochs=3000, verbose=0)

    # Starts: warm start + random points drawn from within the local ball
    starts = [warm_start] + \
             [np.random.uniform(lo, hi).astype(np.float32) for _ in range(n_random_starts)]

    best_val = -np.inf
    best_x_out = warm_start.copy()

    for start in starts:
        x = tf.Variable(start.reshape(1, -1))
        opt = tf.keras.optimizers.Adam(lr)
        for _ in range(n_steps):
            with tf.GradientTape() as tape:
                loss = -model(x)  # negate: minimise negative = maximise output
            grad = tape.gradient(loss, x)
            opt.apply_gradients([(grad, x)])
            # Clip to local ball (not full [0,1]) " prevents degenerate boundary solutions
            x.assign(tf.clip_by_value(x, lo, hi))
        val = float(model(x).numpy()[0][0])
        if val > best_val:
            best_val = val
            best_x_out = x.numpy()[0].copy()

    return best_x_out

np.random.seed(42)
tf.random.set_seed(42)

# Warm-start inputs and search radii for NN gradient ascent
nn_config = {
    5: {'warm_start': [0.274189, 0.896480, 0.929484, 0.928516],           'radius': 0.05},
    6: {'warm_start': [0.386227, 0.370287, 0.535411, 0.895749, 0.315540], 'radius': 0.10},
}

for i in range(1, 9):
    X = np.vstack([
        functions[i]['inputs'],
        round_1[i]['input'],
        round_2[i]['input'],
        round_3[i]['input']
    ])
    y = np.concatenate([
        functions[i]['outputs'],
        [round_1[i]['output']],
        [round_2[i]['output']],
        [round_3[i]['output']]
    ])
    n_dims = X.shape[1]

    if i == 1:
        # Maximin: outputs ~0, no surrogate can help " explore furthest from all data
        candidates = np.random.uniform(0, 1, (100000, n_dims))
        dists = np.min(np.linalg.norm(candidates[:, None, :] - X[None, :, :], axis=2), axis=1)
        query = candidates[np.argmax(dists)]
        method = 'maximin exploration'

    elif i in [5, 6]:
        # NN surrogate + gradient ascent " constrained to local ball around best known input
        cfg = nn_config[i]
        query = nn_gradient_ascent_query(X, y, cfg['warm_start'], cfg['radius'])
        method = f"NN surrogate + gradient ascent (radius={cfg['radius']})"

    else:
        # GP ensemble + LHS " validated in Round 3, working well
        query = gp_ensemble_query(X, y, n_dims)
        method = 'GP ensemble + LHS'

    print(f'--- Function {i} ({method}) ---')
    print(f'Suggested query: {query}')
    print(f'Formatted:       {"-".join(f"{v:.6f}" for v in query)}')
    print()

## Round 5 " Refined Hybrid Strategy
Changes from Round 4:

- **Function 1** " Fixed query `[0.25, 0.25]`: sinusoidal hypothesis " all outputs ~0 regardless of surrogate; deliberately target unexplored lower-left quadrant where a low-frequency wave peak could plausibly sit
- **Functions 2, 4, 8** " GP ensemble + LHS + beta=1.0: all improved in Round 4, no reason to change
- **Function 3** " GP ensemble + LHS + beta=0.5: R4 regressed slightly " reduce beta to force tighter exploitation around R3 best `[0.432, 0.423, 0.508]`
- **Function 5** " NN surrogate + gradient ascent, radius tightened to 0.03 from R4 best `[0.324, 0.946, 0.979, 0.979]`: massive R4 jump suggests we are near the peak " refine precisely
- **Function 6** " NN surrogate + gradient ascent, warm start reverted to R2 best `[0.386, 0.370, 0.535, 0.896, 0.316]`, radius 0.08: R2 remains all-time best for F6 " NN drifted away in R4, pull it back
- **Function 7** " GP ensemble + LHS + beta=0.5: two consecutive regressions " force exploitation tightly around R3 best `[0.085, 0.293, 0.173, 0.127, 0.393, 0.660]`

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel as C
from scipy.stats.qmc import LatinHypercube

# Round 1 results
round_1 = {
    1: {'input': [0.781024, 0.783000],                                                              'output': 3.60011e-33},
    2: {'input': [0.752637, 0.975656],                                                              'output': 0.3504834},
    3: {'input': [0.542581, 0.661593, 0.390176],                                                    'output': -0.0234504},
    4: {'input': [0.627766, 0.478772, 0.475826, 0.299007],                                         'output': -6.5008454},
    5: {'input': [0.274189, 0.896480, 0.929484, 0.928516],                                         'output': 1961.0955629},
    6: {'input': [0.778186, 0.204693, 0.782552, 0.743997, 0.106401],                               'output': -0.6397542},
    7: {'input': [0.107896, 0.541672, 0.297422, 0.268118, 0.470428, 0.780970],                     'output': 1.0204860},
    8: {'input': [0.106447, 0.115956, 0.072929, 0.088786, 0.453935, 0.851055, 0.538307, 0.943085], 'output': 9.5614453},
}

# Round 2 results
round_2 = {
    1: {'input': [0.722566, 0.786336],                                                              'output': -5.8376561e-24},
    2: {'input': [0.338518, 0.489826],                                                              'output': 0.0381634},
    3: {'input': [0.353706, 0.737070, 0.756926],                                                    'output': -0.1039563},
    4: {'input': [0.548017, 0.538375, 0.021034, 0.258912],                                         'output': -10.9738436},
    5: {'input': [0.225843, 0.909370, 0.503192, 0.299769],                                         'output': 58.7131993},
    6: {'input': [0.386227, 0.370287, 0.535411, 0.895749, 0.315540],                               'output': -0.4914677},
    7: {'input': [0.089709, 0.438459, 0.180432, 0.183812, 0.354751, 0.831516],                     'output': 1.3918175},
    8: {'input': [0.443569, 0.599844, 0.033920, 0.013473, 0.948748, 0.067015, 0.092241, 0.036099], 'output': 9.2616659},
}

# Round 3 results
round_3 = {
    1: {'input': [0.787451, 0.713163],                                                              'output': -5.8000091e-23},
    2: {'input': [0.666136, 0.924650],                                                              'output': 0.4682318},
    3: {'input': [0.432014, 0.423353, 0.507804],                                                    'output': -0.0177134},
    4: {'input': [0.469321, 0.416892, 0.307101, 0.426050],                                         'output': -1.3139071},
    5: {'input': [0.264682, 0.835123, 0.958851, 0.940666],                                         'output': 1909.4865637},
    6: {'input': [0.263435, 0.869356, 0.592685, 0.722761, 0.544815],                               'output': -1.2691502},
    7: {'input': [0.085295, 0.293495, 0.173384, 0.126868, 0.393363, 0.660213],                     'output': 1.7092281},
    8: {'input': [0.104083, 0.054438, 0.407037, 0.006596, 0.900508, 0.470715, 0.078870, 0.916641], 'output': 9.6947417},
}

# Round 4 results
round_4 = {
    1: {'input': [0.001601, 0.996704],                                                              'output': 0.0},
    2: {'input': [0.728170, 0.898302],                                                              'output': 0.6095595},
    3: {'input': [0.505240, 0.633279, 0.561304],                                                    'output': -0.0249163},
    4: {'input': [0.373386, 0.339923, 0.390620, 0.428872],                                         'output': 0.3289956},
    5: {'input': [0.324189, 0.946480, 0.979484, 0.978516],                                         'output': 3316.4673094},
    6: {'input': [0.286227, 0.270287, 0.435411, 0.995749, 0.215540],                               'output': -0.8393504},
    7: {'input': [0.031708, 0.132872, 0.302691, 0.030490, 0.427264, 0.704727],                     'output': 1.2439752},
    8: {'input': [0.179513, 0.337380, 0.030860, 0.509674, 0.636042, 0.713639, 0.138762, 0.701544], 'output': 9.7257785},
}

# GP ensemble kernels
gp_kernels = [
    C(1.0) * RBF(length_scale=1.0,    length_scale_bounds=(1e-6, 1e3)),
    C(1.0) * Matern(length_scale=1.0, length_scale_bounds=(1e-6, 1e3), nu=2.5),
    C(1.0) * Matern(length_scale=1.0, length_scale_bounds=(1e-6, 1e3), nu=1.5),
]

n_candidates = 10000
beta_config = {1: 1.0, 2: 1.0, 3: 0.5, 4: 1.0, 5: 1.0, 6: 1.0, 7: 0.5, 8: 1.0}

nn_config = {
    5: {'warm_start': [0.324189, 0.946480, 0.979484, 0.978516], 'radius': 0.03},
    6: {'warm_start': [0.386227, 0.370287, 0.535411, 0.895749, 0.315540], 'radius': 0.08},
}

def gp_ensemble_query(X, y, n_dims, beta=1.0):
    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = (y - y_mean) / y_std
    sampler = LatinHypercube(d=n_dims)
    candidates = sampler.random(n=n_candidates)
    ensemble_ucb = np.zeros(n_candidates)
    for kernel in gp_kernels:
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=False)
        gp.fit(X, y_norm)
        mu, sigma = gp.predict(candidates, return_std=True)
        ensemble_ucb += mu + beta * sigma
    ensemble_ucb /= len(gp_kernels)
    return candidates[np.argmax(ensemble_ucb)]

def nn_gradient_ascent_query(X, y, warm_start, radius, n_random_starts=9, lr=0.05, n_steps=500):
    n_dims = X.shape[1]
    warm_start = np.array(warm_start, dtype=np.float32)
    lo = np.clip(warm_start - radius, 0.0, 1.0)
    hi = np.clip(warm_start + radius, 0.0, 1.0)
    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = ((y - y_mean) / y_std).reshape(-1, 1).astype(np.float32)
    reg = tf.keras.regularizers.L2(0.01)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(8, activation='relu', kernel_regularizer=reg, input_shape=(n_dims,)),
        tf.keras.layers.Dense(8, activation='relu', kernel_regularizer=reg),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse')
    model.fit(X.astype(np.float32), y_norm, epochs=3000, verbose=0)
    starts = [warm_start] + \
             [np.random.uniform(lo, hi).astype(np.float32) for _ in range(n_random_starts)]
    best_val = -np.inf
    best_x_out = warm_start.copy()
    for start in starts:
        x = tf.Variable(start.reshape(1, -1))
        opt = tf.keras.optimizers.Adam(lr)
        for _ in range(n_steps):
            with tf.GradientTape() as tape:
                loss = -model(x)
            grad = tape.gradient(loss, x)
            opt.apply_gradients([(grad, x)])
            x.assign(tf.clip_by_value(x, lo, hi))
        val = float(model(x).numpy()[0][0])
        if val > best_val:
            best_val = val
            best_x_out = x.numpy()[0].copy()
    return best_x_out

np.random.seed(42)
tf.random.set_seed(42)

for i in range(1, 9):
    X = np.vstack([
        functions[i]['inputs'],
        round_1[i]['input'],
        round_2[i]['input'],
        round_3[i]['input'],
        round_4[i]['input'],
    ])
    y = np.concatenate([
        functions[i]['outputs'],
        [round_1[i]['output']],
        [round_2[i]['output']],
        [round_3[i]['output']],
        [round_4[i]['output']],
    ])
    n_dims = X.shape[1]

    if i == 1:
        query = np.array([0.25, 0.25])
        method = 'fixed query " sinusoidal hypothesis [0.25, 0.25]'
    elif i in [5, 6]:
        cfg = nn_config[i]
        query = nn_gradient_ascent_query(X, y, cfg['warm_start'], cfg['radius'])
        method = f"NN surrogate + gradient ascent (radius={cfg['radius']})"
    else:
        query = gp_ensemble_query(X, y, n_dims, beta=beta_config[i])
        method = f"GP ensemble + LHS (beta={beta_config[i]})"

    print(f'--- Function {i} ({method}) ---')
    print(f'Suggested query: {query}')
    print(f'Formatted:       {"-".join(f"{v:.6f}" for v in query)}')
    print()

## Round 6 " Hybrid GP/NN Ensemble Strategy
Key changes from Round 5:

- **All functions except F5** " Hybrid surrogate: weighted ensemble of GP UCB scores and NN predicted values, both normalised to [0,1] before combining. GP contributes uncertainty-aware exploration; NN anchors the search toward regions the accumulated data supports, compensating for GP kernel misfit on complex landscapes.
- **Weights are tuned per function** based on accumulated evidence:
  - GP=0.6 / NN=0.4 for F2 and F8 " GP has been working consistently
  - GP=0.3 / NN=0.7 for F1, F3, F4, F6 " GP has been wandering or producing zero signal
  - GP=0.2 / NN=0.8 for F7 " three consecutive regressions, heaviest NN weighting
- **Function 5** " NN gradient ascent with per-dimension bounds: dims 3 and 4 confirmed at boundary (1.0) in R5 and fixed there; search restricted to dims 1 and 2 only with radius=0.05, effectively reducing F5 to a 2D optimisation problem

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel as C
from scipy.stats.qmc import LatinHypercube

# Round 1 results
round_1 = {
    1: {'input': [0.781024, 0.783000],                                                              'output': 3.60011e-33},
    2: {'input': [0.752637, 0.975656],                                                              'output': 0.3504834},
    3: {'input': [0.542581, 0.661593, 0.390176],                                                    'output': -0.0234504},
    4: {'input': [0.627766, 0.478772, 0.475826, 0.299007],                                         'output': -6.5008454},
    5: {'input': [0.274189, 0.896480, 0.929484, 0.928516],                                         'output': 1961.0955629},
    6: {'input': [0.778186, 0.204693, 0.782552, 0.743997, 0.106401],                               'output': -0.6397542},
    7: {'input': [0.107896, 0.541672, 0.297422, 0.268118, 0.470428, 0.780970],                     'output': 1.0204860},
    8: {'input': [0.106447, 0.115956, 0.072929, 0.088786, 0.453935, 0.851055, 0.538307, 0.943085], 'output': 9.5614453},
}

# Round 2 results
round_2 = {
    1: {'input': [0.722566, 0.786336],                                                              'output': -5.8376561e-24},
    2: {'input': [0.338518, 0.489826],                                                              'output': 0.0381634},
    3: {'input': [0.353706, 0.737070, 0.756926],                                                    'output': -0.1039563},
    4: {'input': [0.548017, 0.538375, 0.021034, 0.258912],                                         'output': -10.9738436},
    5: {'input': [0.225843, 0.909370, 0.503192, 0.299769],                                         'output': 58.7131993},
    6: {'input': [0.386227, 0.370287, 0.535411, 0.895749, 0.315540],                               'output': -0.4914677},
    7: {'input': [0.089709, 0.438459, 0.180432, 0.183812, 0.354751, 0.831516],                     'output': 1.3918175},
    8: {'input': [0.443569, 0.599844, 0.033920, 0.013473, 0.948748, 0.067015, 0.092241, 0.036099], 'output': 9.2616659},
}

# Round 3 results
round_3 = {
    1: {'input': [0.787451, 0.713163],                                                              'output': -5.8000091e-23},
    2: {'input': [0.666136, 0.924650],                                                              'output': 0.4682318},
    3: {'input': [0.432014, 0.423353, 0.507804],                                                    'output': -0.0177134},
    4: {'input': [0.469321, 0.416892, 0.307101, 0.426050],                                         'output': -1.3139071},
    5: {'input': [0.264682, 0.835123, 0.958851, 0.940666],                                         'output': 1909.4865637},
    6: {'input': [0.263435, 0.869356, 0.592685, 0.722761, 0.544815],                               'output': -1.2691502},
    7: {'input': [0.085295, 0.293495, 0.173384, 0.126868, 0.393363, 0.660213],                     'output': 1.7092281},
    8: {'input': [0.104083, 0.054438, 0.407037, 0.006596, 0.900508, 0.470715, 0.078870, 0.916641], 'output': 9.6947417},
}

# Round 4 results
round_4 = {
    1: {'input': [0.001601, 0.996704],                                                              'output': 0.0},
    2: {'input': [0.728170, 0.898302],                                                              'output': 0.6095595},
    3: {'input': [0.505240, 0.633279, 0.561304],                                                    'output': -0.0249163},
    4: {'input': [0.373386, 0.339923, 0.390620, 0.428872],                                         'output': 0.3289956},
    5: {'input': [0.324189, 0.946480, 0.979484, 0.978516],                                         'output': 3316.4673094},
    6: {'input': [0.286227, 0.270287, 0.435411, 0.995749, 0.215540],                               'output': -0.8393504},
    7: {'input': [0.031708, 0.132872, 0.302691, 0.030490, 0.427264, 0.704727],                     'output': 1.2439752},
    8: {'input': [0.179513, 0.337380, 0.030860, 0.509674, 0.636042, 0.713639, 0.138762, 0.701544], 'output': 9.7257785},
}

# Round 5 results
round_5 = {
    1: {'input': [0.250000, 0.250000],                                                              'output': 9.7977e-42},
    2: {'input': [0.707083, 0.907229],                                                              'output': 0.6169678},
    3: {'input': [0.372278, 0.281724, 0.464231],                                                    'output': -0.0264268},
    4: {'input': [0.415977, 0.352537, 0.433120, 0.452705],                                         'output': 0.0636848},
    5: {'input': [0.354189, 0.976480, 1.000000, 1.000000],                                         'output': 4191.3583685},
    6: {'input': [0.433578, 0.290287, 0.536384, 0.975749, 0.371492],                               'output': -0.6934287},
    7: {'input': [0.071560, 0.386831, 0.174019, 0.062513, 0.468446, 0.799137],                     'output': 0.7439422},
    8: {'input': [0.074863, 0.317416, 0.147056, 0.119067, 0.672307, 0.424989, 0.136937, 0.406485], 'output': 9.9434005},
}

# GP ensemble kernels
gp_kernels = [
    C(1.0) * RBF(length_scale=1.0,    length_scale_bounds=(1e-6, 1e3)),
    C(1.0) * Matern(length_scale=1.0, length_scale_bounds=(1e-6, 1e3), nu=2.5),
    C(1.0) * Matern(length_scale=1.0, length_scale_bounds=(1e-6, 1e3), nu=1.5),
]

n_candidates = 10000

def build_nn(n_dims):
    """Small MLP surrogate with L2 regularisation."""
    reg = tf.keras.regularizers.L2(0.01)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=reg, input_shape=(n_dims,)),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=reg),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse')
    return model

def normalise_scores(arr):
    """Normalise array to [0,1] for fair ensemble weighting."""
    arr_min, arr_max = arr.min(), arr.max()
    if arr_max - arr_min < 1e-8:
        return np.zeros_like(arr)
    return (arr - arr_min) / (arr_max - arr_min)

def hybrid_ensemble_query(X, y, n_dims, beta=1.0, gp_weight=0.5, nn_weight=0.5):
    """Hybrid surrogate: weighted ensemble of GP UCB and NN prediction.
    Both scores normalised to [0,1] before combining to ensure fair weighting."""
    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = (y - y_mean) / y_std

    # LHS candidates
    sampler = LatinHypercube(d=n_dims)
    candidates = sampler.random(n=n_candidates)

    # GP ensemble UCB scores
    gp_scores = np.zeros(n_candidates)
    for kernel in gp_kernels:
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=False)
        gp.fit(X, y_norm)
        mu, sigma = gp.predict(candidates, return_std=True)
        gp_scores += mu + beta * sigma
    gp_scores /= len(gp_kernels)

    # NN predicted values
    model = build_nn(n_dims)
    model.fit(X.astype(np.float32), y_norm.reshape(-1, 1).astype(np.float32), epochs=2000, verbose=0)
    nn_scores = model.predict(candidates.astype(np.float32), verbose=0).flatten()

    # Normalise both to [0,1] and combine
    combined = gp_weight * normalise_scores(gp_scores) + nn_weight * normalise_scores(nn_scores)
    return candidates[np.argmax(combined)]

def nn_gradient_ascent_f5(X, y, warm_start, lo, hi, n_random_starts=9, lr=0.05, n_steps=500):
    """NN gradient ascent for F5 with per-dimension bounds.
    Dims 3 and 4 are fixed at 1.0 " confirmed at boundary in R5.
    Search is restricted to dims 1 and 2 only."""
    n_dims = X.shape[1]
    warm_start = np.array(warm_start, dtype=np.float32)
    lo = np.array(lo, dtype=np.float32)
    hi = np.array(hi, dtype=np.float32)

    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = ((y - y_mean) / y_std).reshape(-1, 1).astype(np.float32)

    model = build_nn(n_dims)
    model.fit(X.astype(np.float32), y_norm, epochs=3000, verbose=0)

    starts = [warm_start] + \
             [np.clip(np.random.uniform(lo, hi), lo, hi).astype(np.float32) for _ in range(n_random_starts)]

    best_val = -np.inf
    best_x_out = warm_start.copy()

    for start in starts:
        x = tf.Variable(start.reshape(1, -1))
        opt = tf.keras.optimizers.Adam(lr)
        for _ in range(n_steps):
            with tf.GradientTape() as tape:
                loss = -model(x)
            grad = tape.gradient(loss, x)
            opt.apply_gradients([(grad, x)])
            x.assign(tf.clip_by_value(x, lo, hi))
        val = float(model(x).numpy()[0][0])
        if val > best_val:
            best_val = val
            best_x_out = x.numpy()[0].copy()

    return best_x_out

np.random.seed(42)
tf.random.set_seed(42)

# Per-function GP/NN weights " higher NN weight for consistently failing functions
weight_config = {
    1: (0.3, 0.7),   # GP zero signal across 5 rounds
    2: (0.6, 0.4),   # GP working well
    3: (0.3, 0.7),   # Two consecutive regressions
    4: (0.3, 0.7),   # Significant R5 drop
    6: (0.3, 0.7),   # Five rounds without beating R2 best
    7: (0.2, 0.8),   # Three consecutive regressions " heaviest NN weighting
    8: (0.6, 0.4),   # Consistent GP improvement
}

# F5 " per-dimension bounds: dims 1,2 search radius=0.05, dims 3,4 fixed at 1.0
f5_warm_start = [0.354189, 0.976480, 1.000000, 1.000000]
f5_lo         = [0.304189, 0.926480, 1.000000, 1.000000]
f5_hi         = [0.404189, 1.000000, 1.000000, 1.000000]

for i in range(1, 9):
    X = np.vstack([
        functions[i]['inputs'],
        round_1[i]['input'],
        round_2[i]['input'],
        round_3[i]['input'],
        round_4[i]['input'],
        round_5[i]['input'],
    ])
    y = np.concatenate([
        functions[i]['outputs'],
        [round_1[i]['output']],
        [round_2[i]['output']],
        [round_3[i]['output']],
        [round_4[i]['output']],
        [round_5[i]['output']],
    ])
    n_dims = X.shape[1]

    if i == 5:
        query = nn_gradient_ascent_f5(X, y, f5_warm_start, f5_lo, f5_hi)
        method = 'NN gradient ascent " dims 3,4 fixed at 1.0, radius=0.05 on dims 1,2'
    else:
        gp_w, nn_w = weight_config[i]
        query = hybrid_ensemble_query(X, y, n_dims, beta=1.0, gp_weight=gp_w, nn_weight=nn_w)
        method = f'Hybrid ensemble (GP={gp_w}, NN={nn_w})'

    print(f'--- Function {i} ({method}) ---')
    print(f'Suggested query: {query}')
    print(f'Formatted:       {"-".join(f"{v:.6f}" for v in query)}')
    print()

## Round 7 " Precision Exploitation with NN Gradient Ascent

R6 results: 3 new bests (F5 *' 4544, F6 *' -0.178, F7 *' 2.038). F1 confirmed identically zero (1.7e-191). F2 and F4 regressed badly " hybrid ensemble wandered too far.

With 16 data points per function (10 initial + 6 rounds), the good regions are now well characterised. Round 7 shifts entirely to tight local exploitation.

Key changes from Round 6:

- **Function 1** " Fixed query `[0.5, 0.5]`: confirmed zero across all 6 rounds covering every region of 2D space; no optimisation effort expended
- **Functions 2, 3, 6, 7, 8** " NN gradient ascent from all-time best, tight radius: hybrid ensemble wandered for F2 (0.617 *' -0.045) and F4 (0.329 *' -2.17); constraining search to a small ball around the confirmed best prevents surrogate from escaping the good region
- **Function 4** " Radius=0.02 (smallest): landscape is extremely sensitive " R6 query sat close to R4 best but collapsed output from 0.329 to -2.17; only micro-perturbations are safe
- **Function 5** " NN gradient ascent, dim 1 search extended to [0.354, 0.554]: R6 hit the upper bound at dim 1=0.404 and still improved (4544 vs 4191); gradient is pointing up, push the ceiling higher; dims 2, 3, 4 fixed at 1.0

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import numpy as np

# Round 1 results
round_1 = {
    1: {'input': [0.781024, 0.783000],                                                              'output': 3.60011e-33},
    2: {'input': [0.752637, 0.975656],                                                              'output': 0.3504834},
    3: {'input': [0.542581, 0.661593, 0.390176],                                                    'output': -0.0234504},
    4: {'input': [0.627766, 0.478772, 0.475826, 0.299007],                                         'output': -6.5008454},
    5: {'input': [0.274189, 0.896480, 0.929484, 0.928516],                                         'output': 1961.0955629},
    6: {'input': [0.778186, 0.204693, 0.782552, 0.743997, 0.106401],                               'output': -0.6397542},
    7: {'input': [0.107896, 0.541672, 0.297422, 0.268118, 0.470428, 0.780970],                     'output': 1.0204860},
    8: {'input': [0.106447, 0.115956, 0.072929, 0.088786, 0.453935, 0.851055, 0.538307, 0.943085], 'output': 9.5614453},
}

# Round 2 results
round_2 = {
    1: {'input': [0.722566, 0.786336],                                                              'output': -5.8376561e-24},
    2: {'input': [0.338518, 0.489826],                                                              'output': 0.0381634},
    3: {'input': [0.353706, 0.737070, 0.756926],                                                    'output': -0.1039563},
    4: {'input': [0.548017, 0.538375, 0.021034, 0.258912],                                         'output': -10.9738436},
    5: {'input': [0.225843, 0.909370, 0.503192, 0.299769],                                         'output': 58.7131993},
    6: {'input': [0.386227, 0.370287, 0.535411, 0.895749, 0.315540],                               'output': -0.4914677},
    7: {'input': [0.089709, 0.438459, 0.180432, 0.183812, 0.354751, 0.831516],                     'output': 1.3918175},
    8: {'input': [0.443569, 0.599844, 0.033920, 0.013473, 0.948748, 0.067015, 0.092241, 0.036099], 'output': 9.2616659},
}

# Round 3 results
round_3 = {
    1: {'input': [0.787451, 0.713163],                                                              'output': -5.8000091e-23},
    2: {'input': [0.666136, 0.924650],                                                              'output': 0.4682318},
    3: {'input': [0.432014, 0.423353, 0.507804],                                                    'output': -0.0177134},
    4: {'input': [0.469321, 0.416892, 0.307101, 0.426050],                                         'output': -1.3139071},
    5: {'input': [0.264682, 0.835123, 0.958851, 0.940666],                                         'output': 1909.4865637},
    6: {'input': [0.263435, 0.869356, 0.592685, 0.722761, 0.544815],                               'output': -1.2691502},
    7: {'input': [0.085295, 0.293495, 0.173384, 0.126868, 0.393363, 0.660213],                     'output': 1.7092281},
    8: {'input': [0.104083, 0.054438, 0.407037, 0.006596, 0.900508, 0.470715, 0.078870, 0.916641], 'output': 9.6947417},
}

# Round 4 results
round_4 = {
    1: {'input': [0.001601, 0.996704],                                                              'output': 0.0},
    2: {'input': [0.728170, 0.898302],                                                              'output': 0.6095595},
    3: {'input': [0.505240, 0.633279, 0.561304],                                                    'output': -0.0249163},
    4: {'input': [0.373386, 0.339923, 0.390620, 0.428872],                                         'output': 0.3289956},
    5: {'input': [0.324189, 0.946480, 0.979484, 0.978516],                                         'output': 3316.4673094},
    6: {'input': [0.286227, 0.270287, 0.435411, 0.995749, 0.215540],                               'output': -0.8393504},
    7: {'input': [0.031708, 0.132872, 0.302691, 0.030490, 0.427264, 0.704727],                     'output': 1.2439752},
    8: {'input': [0.179513, 0.337380, 0.030860, 0.509674, 0.636042, 0.713639, 0.138762, 0.701544], 'output': 9.7257785},
}

# Round 5 results
round_5 = {
    1: {'input': [0.250000, 0.250000],                                                              'output': 9.7977e-42},
    2: {'input': [0.707083, 0.907229],                                                              'output': 0.6169678},
    3: {'input': [0.372278, 0.281724, 0.464231],                                                    'output': -0.0264268},
    4: {'input': [0.415977, 0.352537, 0.433120, 0.452705],                                         'output': 0.0636848},
    5: {'input': [0.354189, 0.976480, 1.000000, 1.000000],                                         'output': 4191.3583685},
    6: {'input': [0.433578, 0.290287, 0.536384, 0.975749, 0.371492],                               'output': -0.6934287},
    7: {'input': [0.071560, 0.386831, 0.174019, 0.062513, 0.468446, 0.799137],                     'output': 0.7439422},
    8: {'input': [0.074863, 0.317416, 0.147056, 0.119067, 0.672307, 0.424989, 0.136937, 0.406485], 'output': 9.9434005},
}

# Round 6 results
round_6 = {
    1: {'input': [0.998394, 0.999482],                                                              'output': 1.6996e-191},
    2: {'input': [0.944177, 0.013586],                                                              'output': -0.0445381},
    3: {'input': [0.990867, 0.502049, 0.536263],                                                    'output': -0.0184090},
    4: {'input': [0.385593, 0.396692, 0.459001, 0.481881],                                         'output': -2.1657928},
    5: {'input': [0.404189, 1.000000, 1.000000, 1.000000],                                         'output': 4544.4767958},
    6: {'input': [0.411313, 0.361437, 0.769565, 0.849107, 0.184824],                               'output': -0.1778310},
    7: {'input': [0.099423, 0.293146, 0.092186, 0.195886, 0.363972, 0.648441],                     'output': 2.0376374},
    8: {'input': [0.025950, 0.163720, 0.116687, 0.391130, 0.980564, 0.689047, 0.147088, 0.049991], 'output': 9.8422788},
}

def build_nn(n_dims):
    """Small MLP surrogate with L2 regularisation."""
    reg = tf.keras.regularizers.L2(0.01)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=reg, input_shape=(n_dims,)),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=reg),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse')
    return model

def nn_gradient_ascent_query(X, y, lo, hi, warm_start, n_random_starts=9, lr=0.05, n_steps=500):
    """NN surrogate + gradient ascent, constrained to [lo, hi] per dimension.
    Warm start is tried first; n_random_starts additional starts drawn from within the bounds."""
    n_dims = X.shape[1]
    lo = np.array(lo, dtype=np.float32)
    hi = np.array(hi, dtype=np.float32)
    warm_start = np.array(warm_start, dtype=np.float32)

    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = ((y - y_mean) / y_std).reshape(-1, 1).astype(np.float32)

    model = build_nn(n_dims)
    model.fit(X.astype(np.float32), y_norm, epochs=3000, verbose=0)

    # Warm start + random restarts within the search ball
    starts = [warm_start] + \
             [np.random.uniform(lo, hi).astype(np.float32) for _ in range(n_random_starts)]

    best_val  = -np.inf
    best_x    = warm_start.copy()

    for start in starts:
        x = tf.Variable(start.reshape(1, -1))
        opt = tf.keras.optimizers.Adam(lr)
        for _ in range(n_steps):
            with tf.GradientTape() as tape:
                loss = -model(x)
            grad = tape.gradient(loss, x)
            opt.apply_gradients([(grad, x)])
            x.assign(tf.clip_by_value(x, lo, hi))
        val = float(model(x).numpy()[0][0])
        if val > best_val:
            best_val = val
            best_x   = x.numpy()[0].copy()

    return best_x

np.random.seed(42)
tf.random.set_seed(42)

# All-time best inputs (used as warm starts) and per-dimension search bounds
# lo/hi = warm_start +/- radius, clipped to [0, 1]
nn_config = {
    # F2: R5 best [0.707, 0.907] " radius 0.05
    2: {'warm': [0.707083, 0.907229],
        'lo':   [0.657083, 0.857229],
        'hi':   [0.757083, 0.957229]},
    # F3: R3 best [0.432, 0.423, 0.508] " radius 0.03
    3: {'warm': [0.432014, 0.423353, 0.507804],
        'lo':   [0.402014, 0.393353, 0.477804],
        'hi':   [0.462014, 0.453353, 0.537804]},
    # F4: R4 best [0.373, 0.340, 0.391, 0.429] " radius 0.02 (very sensitive landscape)
    4: {'warm': [0.373386, 0.339923, 0.390620, 0.428872],
        'lo':   [0.353386, 0.319923, 0.370620, 0.408872],
        'hi':   [0.393386, 0.359923, 0.410620, 0.448872]},
    # F5: R6 best [0.404, 1.0, 1.0, 1.0] " dim 1 search [0.354, 0.554], dims 2-4 fixed at 1.0
    5: {'warm': [0.404189, 1.000000, 1.000000, 1.000000],
        'lo':   [0.354189, 1.000000, 1.000000, 1.000000],
        'hi':   [0.554189, 1.000000, 1.000000, 1.000000]},
    # F6: R6 best [0.411, 0.361, 0.770, 0.849, 0.185] " radius 0.06
    6: {'warm': [0.411313, 0.361437, 0.769565, 0.849107, 0.184824],
        'lo':   [0.351313, 0.301437, 0.709565, 0.789107, 0.124824],
        'hi':   [0.471313, 0.421437, 0.829565, 0.909107, 0.244824]},
    # F7: R6 best [0.099, 0.293, 0.092, 0.196, 0.364, 0.648] " radius 0.05
    7: {'warm': [0.099423, 0.293146, 0.092186, 0.195886, 0.363972, 0.648441],
        'lo':   [0.049423, 0.243146, 0.042186, 0.145886, 0.313972, 0.598441],
        'hi':   [0.149423, 0.343146, 0.142186, 0.245886, 0.413972, 0.698441]},
    # F8: R5 best [0.075, 0.317, 0.147, 0.119, 0.672, 0.425, 0.137, 0.406] " radius 0.05
    8: {'warm': [0.074863, 0.317416, 0.147056, 0.119067, 0.672307, 0.424989, 0.136937, 0.406485],
        'lo':   [0.024863, 0.267416, 0.097056, 0.069067, 0.622307, 0.374989, 0.086937, 0.356485],
        'hi':   [0.124863, 0.367416, 0.197056, 0.169067, 0.722307, 0.474989, 0.186937, 0.456485]},
}

for i in range(1, 9):
    X = np.vstack([
        functions[i]['inputs'],
        round_1[i]['input'],
        round_2[i]['input'],
        round_3[i]['input'],
        round_4[i]['input'],
        round_5[i]['input'],
        round_6[i]['input'],
    ])
    y = np.concatenate([
        functions[i]['outputs'],
        [round_1[i]['output']],
        [round_2[i]['output']],
        [round_3[i]['output']],
        [round_4[i]['output']],
        [round_5[i]['output']],
        [round_6[i]['output']],
    ])

    if i == 1:
        # Confirmed zero " no optimisation value. Submit centre of domain.
        query  = np.array([0.5, 0.5])
        method = 'fixed [0.5, 0.5] " function confirmed identically zero'
    else:
        cfg    = nn_config[i]
        query  = nn_gradient_ascent_query(X, y, cfg['lo'], cfg['hi'], cfg['warm'])
        method = f"NN gradient ascent " warm={cfg['warm']}, radius varies per dim"

    print(f'--- Function {i} ({method}) ---')
    print(f'Suggested query: {query}')
    print(f'Formatted:       {"-".join(f"{v:.6f}" for v in query)}')
    print()

## Round 8 " All-Time Best Recovery

R7 results: F2 and F5 set new all-time bests. F2 reached 0.6686 at [0.715128, 0.911455]; F5 reached 4797 continuing the dim-1 boundary climb. F1 returned 2.7e-9 " effectively zero. F3, F6, F7, and F8 all regressed vs their all-time bests " the tight NN search drifted in unhelpful directions.

All-time bests going into Round 8:

| Fn | Best output | Best input (round) |
|----|-------------|--------------------|
| 1  | ~0 (skip)   | " |
| 2  | 0.6686      | R7: [0.715, 0.911] |
| 3  | ^'0.0177     | R3: [0.432, 0.423, 0.508] |
| 4  | 0.3290      | R4: [0.373, 0.340, 0.391, 0.429] |
| 5  | 4797        | R7: [0.554, 1.0, 1.0, 1.0] |
| 6  | ^'0.1778     | R6: [0.411, 0.361, 0.770, 0.849, 0.185] |
| 7  | 2.0376      | R6: [0.099, 0.293, 0.092, 0.196, 0.364, 0.648] |
| 8  | 9.9434      | R5: [0.075, 0.317, 0.147, 0.119, 0.672, 0.425, 0.137, 0.406] |

Round 8 strategy " pure exploitation back to confirmed good regions:

- **Function 1** " Fixed [0.95, 0.05]: zero confirmed across every sampled region; last unexplored corner
- **Function 2** " NN gradient ascent warm-started at R7 best [0.715, 0.911], radius 0.03
- **Function 3** " NN gradient ascent returning to R3 best [0.432, 0.423, 0.508], radius 0.01: R7 regressed by +87%
- **Function 4** " NN gradient ascent searching below R4 best on dims 1-3 (decrease by 0.02): R7 tried increasing and regressed " try opposite direction
- **Function 5** " NN gradient ascent, push dim1 toward 0.75 with dims 2-4 fixed at 1.0: consistent monotone improvement each round along this boundary
- **Function 6** " NN gradient ascent warm-started at the mirror of R7 from R6 best: R7 moved +0.06 on dims 3-4 and collapsed output; reverse that direction
- **Function 7** " NN gradient ascent returning to R6 best with dims 1-2 nudged upward by 0.01
- **Function 8** " GP ensemble UCB restricted to local ball (radius 0.05) around R5 best

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel as C
from scipy.stats.qmc import LatinHypercube

# Round 1 results
round_1 = {
    1: {'input': [0.781024, 0.783000],                                                              'output': 3.60011e-33},
    2: {'input': [0.752637, 0.975656],                                                              'output': 0.3504834},
    3: {'input': [0.542581, 0.661593, 0.390176],                                                    'output': -0.0234504},
    4: {'input': [0.627766, 0.478772, 0.475826, 0.299007],                                         'output': -6.5008454},
    5: {'input': [0.274189, 0.896480, 0.929484, 0.928516],                                         'output': 1961.0955629},
    6: {'input': [0.778186, 0.204693, 0.782552, 0.743997, 0.106401],                               'output': -0.6397542},
    7: {'input': [0.107896, 0.541672, 0.297422, 0.268118, 0.470428, 0.780970],                     'output': 1.0204860},
    8: {'input': [0.106447, 0.115956, 0.072929, 0.088786, 0.453935, 0.851055, 0.538307, 0.943085], 'output': 9.5614453},
}

# Round 2 results
round_2 = {
    1: {'input': [0.722566, 0.786336],                                                              'output': -5.8376561e-24},
    2: {'input': [0.338518, 0.489826],                                                              'output': 0.0381634},
    3: {'input': [0.353706, 0.737070, 0.756926],                                                    'output': -0.1039563},
    4: {'input': [0.548017, 0.538375, 0.021034, 0.258912],                                         'output': -10.9738436},
    5: {'input': [0.225843, 0.909370, 0.503192, 0.299769],                                         'output': 58.7131993},
    6: {'input': [0.386227, 0.370287, 0.535411, 0.895749, 0.315540],                               'output': -0.4914677},
    7: {'input': [0.089709, 0.438459, 0.180432, 0.183812, 0.354751, 0.831516],                     'output': 1.3918175},
    8: {'input': [0.443569, 0.599844, 0.033920, 0.013473, 0.948748, 0.067015, 0.092241, 0.036099], 'output': 9.2616659},
}

# Round 3 results
round_3 = {
    1: {'input': [0.787451, 0.713163],                                                              'output': -5.8000091e-23},
    2: {'input': [0.666136, 0.924650],                                                              'output': 0.4682318},
    3: {'input': [0.432014, 0.423353, 0.507804],                                                    'output': -0.0177134},
    4: {'input': [0.469321, 0.416892, 0.307101, 0.426050],                                         'output': -1.3139071},
    5: {'input': [0.264682, 0.835123, 0.958851, 0.940666],                                         'output': 1909.4865637},
    6: {'input': [0.263435, 0.869356, 0.592685, 0.722761, 0.544815],                               'output': -1.2691502},
    7: {'input': [0.085295, 0.293495, 0.173384, 0.126868, 0.393363, 0.660213],                     'output': 1.7092281},
    8: {'input': [0.104083, 0.054438, 0.407037, 0.006596, 0.900508, 0.470715, 0.078870, 0.916641], 'output': 9.6947417},
}

# Round 4 results
round_4 = {
    1: {'input': [0.001601, 0.996704],                                                              'output': 0.0},
    2: {'input': [0.728170, 0.898302],                                                              'output': 0.6095595},
    3: {'input': [0.505240, 0.633279, 0.561304],                                                    'output': -0.0249163},
    4: {'input': [0.373386, 0.339923, 0.390620, 0.428872],                                         'output': 0.3289956},
    5: {'input': [0.324189, 0.946480, 0.979484, 0.978516],                                         'output': 3316.4673094},
    6: {'input': [0.286227, 0.270287, 0.435411, 0.995749, 0.215540],                               'output': -0.8393504},
    7: {'input': [0.031708, 0.132872, 0.302691, 0.030490, 0.427264, 0.704727],                     'output': 1.2439752},
    8: {'input': [0.179513, 0.337380, 0.030860, 0.509674, 0.636042, 0.713639, 0.138762, 0.701544], 'output': 9.7257785},
}

# Round 5 results
round_5 = {
    1: {'input': [0.250000, 0.250000],                                                              'output': 9.7977e-42},
    2: {'input': [0.707083, 0.907229],                                                              'output': 0.6169678},
    3: {'input': [0.372278, 0.281724, 0.464231],                                                    'output': -0.0264268},
    4: {'input': [0.415977, 0.352537, 0.433120, 0.452705],                                         'output': 0.0636848},
    5: {'input': [0.354189, 0.976480, 1.000000, 1.000000],                                         'output': 4191.3583685},
    6: {'input': [0.433578, 0.290287, 0.536384, 0.975749, 0.371492],                               'output': -0.6934287},
    7: {'input': [0.071560, 0.386831, 0.174019, 0.062513, 0.468446, 0.799137],                     'output': 0.7439422},
    8: {'input': [0.074863, 0.317416, 0.147056, 0.119067, 0.672307, 0.424989, 0.136937, 0.406485], 'output': 9.9434005},
}

# Round 6 results
round_6 = {
    1: {'input': [0.998394, 0.999482],                                                              'output': 1.6996e-191},
    2: {'input': [0.944177, 0.013586],                                                              'output': -0.0445381},
    3: {'input': [0.990867, 0.502049, 0.536263],                                                    'output': -0.0184090},
    4: {'input': [0.385593, 0.396692, 0.459001, 0.481881],                                         'output': -2.1657928},
    5: {'input': [0.404189, 1.000000, 1.000000, 1.000000],                                         'output': 4544.4767958},
    6: {'input': [0.411313, 0.361437, 0.769565, 0.849107, 0.184824],                               'output': -0.1778310},
    7: {'input': [0.099423, 0.293146, 0.092186, 0.195886, 0.363972, 0.648441],                     'output': 2.0376374},
    8: {'input': [0.025950, 0.163720, 0.116687, 0.391130, 0.980564, 0.689047, 0.147088, 0.049991], 'output': 9.8422788},
}

# Round 7 results
round_7 = {
    1: {'input': [0.500000, 0.500000],                                                              'output': 2.6752879910742468e-09},
    2: {'input': [0.715128, 0.911455],                                                              'output': 0.6685636148734443},
    3: {'input': [0.462014, 0.393353, 0.537804],                                                    'output': -0.03312160950565687},
    4: {'input': [0.368239, 0.359923, 0.404251, 0.448872],                                         'output': 0.13723939198642343},
    5: {'input': [0.554189, 1.000000, 1.000000, 1.000000],                                         'output': 4797.241552298715},
    6: {'input': [0.394491, 0.365429, 0.829565, 0.909107, 0.124826],                               'output': -0.3587402287596108},
    7: {'input': [0.049423, 0.243146, 0.042186, 0.145886, 0.313972, 0.676699],                     'output': 1.8153615177642732},
    8: {'input': [0.024863, 0.267416, 0.097056, 0.069067, 0.722307, 0.474989, 0.086937, 0.356485], 'output': 9.929976182803},
}

def build_nn(n_dims):
    """Small MLP surrogate with L2 regularisation."""
    reg = tf.keras.regularizers.L2(0.01)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=reg, input_shape=(n_dims,)),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=reg),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse')
    return model

def nn_gradient_ascent_query(X, y, lo, hi, warm_start, n_random_starts=9, lr=0.05, n_steps=500):
    """NN surrogate + gradient ascent, constrained to [lo, hi] per dimension."""
    n_dims = X.shape[1]
    lo = np.array(lo, dtype=np.float32)
    hi = np.array(hi, dtype=np.float32)
    warm_start = np.array(warm_start, dtype=np.float32)

    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = ((y - y_mean) / y_std).reshape(-1, 1).astype(np.float32)

    model = build_nn(n_dims)
    model.fit(X.astype(np.float32), y_norm, epochs=3000, verbose=0)

    starts = [warm_start] + \
             [np.random.uniform(lo, hi).astype(np.float32) for _ in range(n_random_starts)]

    best_val = -np.inf
    best_x   = warm_start.copy()

    for start in starts:
        x = tf.Variable(start.reshape(1, -1))
        opt = tf.keras.optimizers.Adam(lr)
        for _ in range(n_steps):
            with tf.GradientTape() as tape:
                loss = -model(x)
            grad = tape.gradient(loss, x)
            opt.apply_gradients([(grad, x)])
            x.assign(tf.clip_by_value(x, lo, hi))
        val = float(model(x).numpy()[0][0])
        if val > best_val:
            best_val = val
            best_x   = x.numpy()[0].copy()

    return best_x

def gp_local_ucb_query(X, y, center, radius, beta=1.0, n_candidates=10000):
    """GP ensemble UCB with candidates restricted to a local ball around center."""
    n_dims = X.shape[1]
    center = np.array(center, dtype=float)
    lo = np.clip(center - radius, 0.0, 1.0)
    hi = np.clip(center + radius, 0.0, 1.0)

    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = (y - y_mean) / y_std

    sampler = LatinHypercube(d=n_dims)
    unit_cands = sampler.random(n=n_candidates)
    candidates = lo + unit_cands * (hi - lo)

    kernels = [
        C(1.0) * RBF(length_scale=1.0,    length_scale_bounds=(1e-6, 1e3)),
        C(1.0) * Matern(length_scale=1.0, length_scale_bounds=(1e-6, 1e3), nu=2.5),
        C(1.0) * Matern(length_scale=1.0, length_scale_bounds=(1e-6, 1e3), nu=1.5),
    ]
    ensemble_ucb = np.zeros(n_candidates)
    for kernel in kernels:
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=False)
        gp.fit(X, y_norm)
        mu, sigma = gp.predict(candidates, return_std=True)
        ensemble_ucb += mu + beta * sigma
    ensemble_ucb /= len(kernels)
    return candidates[np.argmax(ensemble_ucb)]

np.random.seed(42)
tf.random.set_seed(42)

# Per-function NN gradient ascent configs
# F4: search BELOW R4 best on dims 1-3 " R7 tried increasing and regressed; try opposite direction
# F6: warm-start is the mirror of R7 from R6 best (R6 + (R6 - R7)) " reverses the failed direction
nn_config = {
    # F2: R7 best [0.715, 0.911] " radius 0.03
    2: {'warm': [0.715128, 0.911455],
        'lo':   [0.685128, 0.881455],
        'hi':   [0.745128, 0.941455]},
    # F3: return to R3 best [0.432, 0.423, 0.508] " radius 0.01
    3: {'warm': [0.432014, 0.423353, 0.507804],
        'lo':   [0.422014, 0.413353, 0.497804],
        'hi':   [0.442014, 0.433353, 0.517804]},
    # F4: decrease dims 1-3 from R4 best by 0.02; keep dim 4 centred on R4 best
    4: {'warm': [0.353386, 0.319923, 0.370620, 0.428872],
        'lo':   [0.333386, 0.299923, 0.350620, 0.408872],
        'hi':   [0.373386, 0.339923, 0.390620, 0.448872]},
    # F5: push dim1 toward 0.75; dims 2-4 fixed at 1.0 (confirmed boundary)
    5: {'warm': [0.750000, 1.000000, 1.000000, 1.000000],
        'lo':   [0.650000, 1.000000, 1.000000, 1.000000],
        'hi':   [0.900000, 1.000000, 1.000000, 1.000000]},
    # F6: 2*R6_best - R7_query = mirror point " opposite direction from R7's failed move
    6: {'warm': [0.428135, 0.357445, 0.709565, 0.789107, 0.244822],
        'lo':   [0.408135, 0.337445, 0.689565, 0.769107, 0.224822],
        'hi':   [0.448135, 0.377445, 0.729565, 0.809107, 0.264822]},
    # F7: R6 best with dims 1-2 nudged up by 0.01; radius 0.02
    7: {'warm': [0.109423, 0.303146, 0.092186, 0.195886, 0.363972, 0.648441],
        'lo':   [0.089423, 0.283146, 0.072186, 0.175886, 0.343972, 0.628441],
        'hi':   [0.129423, 0.323146, 0.112186, 0.215886, 0.383972, 0.668441]},
}

# F8: GP-UCB around R5 best (all-time best for F8)
f8_center = [0.074863, 0.317416, 0.147056, 0.119067, 0.672307, 0.424989, 0.136937, 0.406485]
f8_radius = 0.05

for i in range(1, 9):
    X = np.vstack([
        functions[i]['inputs'],
        round_1[i]['input'],
        round_2[i]['input'],
        round_3[i]['input'],
        round_4[i]['input'],
        round_5[i]['input'],
        round_6[i]['input'],
        round_7[i]['input'],
    ])
    y = np.concatenate([
        functions[i]['outputs'],
        [round_1[i]['output']],
        [round_2[i]['output']],
        [round_3[i]['output']],
        [round_4[i]['output']],
        [round_5[i]['output']],
        [round_6[i]['output']],
        [round_7[i]['output']],
    ])

    if i == 1:
        query  = np.array([0.95, 0.05])
        method = 'fixed [0.95, 0.05] " last unexplored corner, F1 confirmed zero'
    elif i == 8:
        query  = gp_local_ucb_query(X, y, f8_center, f8_radius)
        method = f'GP ensemble UCB " local ball around R5 best, radius={f8_radius}'
    else:
        cfg    = nn_config[i]
        query  = nn_gradient_ascent_query(X, y, cfg['lo'], cfg['hi'], cfg['warm'])
        method = f"NN gradient ascent " warm={cfg['warm']}"

    print(f'--- Function {i} ({method}) ---')
    print(f'Suggested query: {query}')
    print(f'Formatted:       {"-".join(f"{v:.6f}" for v in query)}')
    print()

## Round 9 " Exploit F5 Boundary + Recover Regressions

R8 results: 3 new bests (F3, F5, F8). F5 headline " pushing dim1 from 0.554 *' 0.9 jumped output from 4797 *' 7073 (+47%). F2, F4, F6, F7 all regressed again.

All-time bests going into Round 9:

| Fn | Best output | Best input (round) |
|----|-------------|--------------------|
| 1  | 0.0 (skip)  | R4: [0.002, 0.997] " confirmed zero, retired |
| 2  | 0.6686      | R7: [0.715, 0.911] |
| 3  | ^'0.0129     | R8: [0.442, 0.433, 0.518] |
| 4  | 0.3290      | R4: [0.373, 0.340, 0.391, 0.429] |
| 5  | 7073.45     | R8: [0.9, 1.0, 1.0, 1.0] |
| 6  | ^'0.1778     | R6: [0.411, 0.361, 0.770, 0.849, 0.185] |
| 7  | 2.0376      | R6: [0.099, 0.293, 0.092, 0.196, 0.364, 0.648] |
| 8  | 9.9654      | R8: [0.054, 0.274, 0.099, 0.151, 0.706, 0.443, 0.170, 0.442] |

Round 9 strategy:

- **Function 1** " Fixed [0.5, 0.5]: confirmed zero, no budget spent
- **Function 2** " NN gradient ascent back to R7 best [0.715, 0.911], radius 0.01: R8 moved dim2 down to 0.881 and collapsed output; dim2 near 0.911 appears critical
- **Function 3** " NN gradient ascent from R8 best [0.442, 0.433, 0.518], radius 0.01: new best achieved in R8, continue tight exploitation
- **Function 4** " NN gradient ascent micro-perturb around R4 best, radius 0.01: every attempt to move away from R4 best has regressed; only micro-perturbation safe
- **Function 5** " Push dim1 to 1.0 (upper boundary), dims 2-4 fixed at 1.0: dim1 gradient has been consistently positive across R6*'R7*'R8 (0.404*'0.554*'0.9*'?); hit the ceiling
- **Function 6** " NN gradient ascent back to R6 best [0.411, 0.361, 0.770, 0.849, 0.185], radius 0.005: R7 and R8 both failed to improve; landscape is extremely sensitive, tightest possible radius
- **Function 7** " NN gradient ascent back to R6 best [0.099, 0.293, 0.092, 0.196, 0.364, 0.648], radius 0.01: R8 warm-started from R7 (already regressed) " need to return to confirmed peak directly
- **Function 8** " GP ensemble UCB around R8 best, radius 0.03: new best in R8, tighten search radius around confirmed improvement

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, ConstantKernel as C

# "" R8 results """"""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
round_8 = {
    1: {'input': np.array([0.950000, 0.050000]),                                                              'output': 4.406064e-291},
    2: {'input': np.array([0.711286, 0.881455]),                                                              'output': 0.527492},
    3: {'input': np.array([0.442014, 0.433337, 0.517804]),                                                    'output': -0.012923},
    4: {'input': np.array([0.373386, 0.339923, 0.390620, 0.436852]),                                         'output': 0.211078},
    5: {'input': np.array([0.900000, 1.000000, 1.000000, 1.000000]),                                         'output': 7073.445506},
    6: {'input': np.array([0.408135, 0.337445, 0.729565, 0.809107, 0.224822]),                               'output': -0.298438},
    7: {'input': np.array([0.089423, 0.283146, 0.072186, 0.185421, 0.380716, 0.628441]),                     'output': 1.870841},
    8: {'input': np.array([0.053794, 0.273991, 0.098763, 0.151316, 0.705994, 0.442635, 0.169673, 0.441635]), 'output': 9.965371},
}

# All-time bests after R8
best_after_r8 = {
    1: {'input': np.array([0.001601, 0.996704]),                                                                  'output': 0.0,      'round': 'R4'},
    2: {'input': np.array([0.715128, 0.911455]),                                                                  'output': 0.6686,   'round': 'R7'},
    3: {'input': np.array([0.442014, 0.433337, 0.517804]),                                                        'output': -0.012923,'round': 'R8'},
    4: {'input': np.array([0.373386, 0.339923, 0.390620, 0.428872]),                                             'output': 0.3290,   'round': 'R4'},
    5: {'input': np.array([0.900000, 1.000000, 1.000000, 1.000000]),                                             'output': 7073.445, 'round': 'R8'},
    6: {'input': np.array([0.411313, 0.361437, 0.769565, 0.849107, 0.184824]),                                   'output': -0.1778,  'round': 'R6'},
    7: {'input': np.array([0.099423, 0.293146, 0.092186, 0.195886, 0.363972, 0.648441]),                         'output': 2.0376,   'round': 'R6'},
    8: {'input': np.array([0.053794, 0.273991, 0.098763, 0.151316, 0.705994, 0.442635, 0.169673, 0.441635]),     'output': 9.965371, 'round': 'R8'},
}

print("All-time bests after R8:")
for i in range(1, 9):
    b = best_after_r8[i]
    print(f"  F{i}: {b['output']:.6g} ({b['round']})")

# "" Shared surrogate helpers """""""""""""""""""""""""""""""""""""""""""""""""""
def nn_gradient_ascent_query(X, y, lo, hi, warm_start, steps=400, lr=0.003):
    lo, hi = np.array(lo), np.array(hi)
    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = (y - y_mean) / y_std

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(X.shape[1],)),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse')
    model.fit(X, y_norm, epochs=300, verbose=0, batch_size=min(16, len(y)))

    x_var = tf.Variable([warm_start], dtype=tf.float32)
    opt   = tf.keras.optimizers.Adam(lr)
    for _ in range(steps):
        with tf.GradientTape() as tape:
            loss = -model(x_var)
        grads = tape.gradient(loss, x_var)
        opt.apply_gradients([(grads, x_var)])
        x_var.assign(tf.clip_by_value(x_var, lo, hi))

    return x_var.numpy()[0]

def gp_local_ucb_query(X, y, center, radius, n_candidates=2000, beta=2.0):
    center = np.array(center)
    lo = np.clip(center - radius, 0, 1)
    hi = np.clip(center + radius, 0, 1)
    candidates = np.random.uniform(lo, hi, (n_candidates, len(center)))

    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = (y - y_mean) / y_std

    kernels = [
        C(1.0) * Matern(length_scale=0.1, nu=2.5),
        C(1.0) * Matern(length_scale=0.3, nu=1.5),
        C(1.0) * RBF(length_scale=0.2),
    ]
    ensemble_ucb = np.zeros(n_candidates)
    for kernel in kernels:
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3, normalize_y=False)
        gp.fit(X, y_norm)
        mu, sigma = gp.predict(candidates, return_std=True)
        ensemble_ucb += mu + beta * sigma
    ensemble_ucb /= len(kernels)
    return candidates[np.argmax(ensemble_ucb)]

np.random.seed(42)
tf.random.set_seed(42)

# "" Per-function R9 configs """"""""""""""""""""""""""""""""""""""""""""""""""""
# F2: return to R7 best " dim2 drop from 0.911*'0.881 in R8 collapsed output
# F3: continue from R8 new best " tight local exploitation
# F4: micro-perturb around R4 best " every departure has regressed; radius 0.01
# F5: push dim1 to 1.0 " gradient consistently positive across R6*'R7*'R8
# F6: return to R6 best with tightest radius (0.005) " R7 and R8 both failed
# F7: return directly to R6 best " R8 warm-started from R7 (already regressed)
nn_config = {
    2: {'warm': [0.715128, 0.911455],
        'lo':   [0.705128, 0.901455],
        'hi':   [0.725128, 0.921455]},
    3: {'warm': [0.442014, 0.433337, 0.517804],
        'lo':   [0.432014, 0.423337, 0.507804],
        'hi':   [0.452014, 0.443337, 0.527804]},
    4: {'warm': [0.373386, 0.339923, 0.390620, 0.428872],
        'lo':   [0.363386, 0.329923, 0.380620, 0.418872],
        'hi':   [0.383386, 0.349923, 0.400620, 0.438872]},
    5: {'warm': [1.000000, 1.000000, 1.000000, 1.000000],
        'lo':   [0.950000, 1.000000, 1.000000, 1.000000],
        'hi':   [1.000000, 1.000000, 1.000000, 1.000000]},
    6: {'warm': [0.411313, 0.361437, 0.769565, 0.849107, 0.184824],
        'lo':   [0.406313, 0.356437, 0.764565, 0.844107, 0.179824],
        'hi':   [0.416313, 0.366437, 0.774565, 0.854107, 0.189824]},
    7: {'warm': [0.099423, 0.293146, 0.092186, 0.195886, 0.363972, 0.648441],
        'lo':   [0.089423, 0.283146, 0.082186, 0.185886, 0.353972, 0.638441],
        'hi':   [0.109423, 0.303146, 0.102186, 0.205886, 0.373972, 0.658441]},
}

# F8: GP-UCB tightened around R8 new best
f8_center = round_8[8]['input']
f8_radius  = 0.03

for i in range(1, 9):
    X = np.vstack([
        functions[i]['inputs'],
        round_1[i]['input'], round_2[i]['input'], round_3[i]['input'],
        round_4[i]['input'], round_5[i]['input'], round_6[i]['input'],
        round_7[i]['input'], round_8[i]['input'],
    ])
    y = np.concatenate([
        functions[i]['outputs'],
        [round_1[i]['output']], [round_2[i]['output']], [round_3[i]['output']],
        [round_4[i]['output']], [round_5[i]['output']], [round_6[i]['output']],
        [round_7[i]['output']], [round_8[i]['output']],
    ])

    if i == 1:
        query  = np.array([0.5, 0.5])
        method = 'fixed [0.5, 0.5] " F1 confirmed zero, no budget spent'
    elif i == 8:
        query  = gp_local_ucb_query(X, y, f8_center, f8_radius)
        method = f'GP ensemble UCB " local ball around R8 best, radius={f8_radius}'
    else:
        cfg    = nn_config[i]
        query  = nn_gradient_ascent_query(X, y, cfg['lo'], cfg['hi'], cfg['warm'])
        method = f"NN gradient ascent " warm={cfg['warm']}"

    print(f'--- Function {i} ({method}) ---')
    print(f'Suggested query: {query}')
    print(f'Formatted:       {"-".join(f"{v:.6f}" for v in query)}')
    print()

## Round 10 -- Anchor Confirmed Bests + Exploit R9 New Highs

R9 results: 2 new bests (F5: 7073 -> 8662, F7: 2.038 -> 2.135). F2, F3, F8 all regressed -- NN surrogate continues drifting away from confirmed peaks.

All-time bests going into Round 10:

| Fn | Best output | Best input (round) |
|----|-------------|--------------------|
| 1  | 0.0 (skip)  | confirmed zero, retired |
| 2  | 0.6686      | R7: [0.715, 0.911] |
| 3  | -0.0129     | R8: [0.442, 0.433, 0.518] |
| 4  | 0.3290      | R4: [0.373, 0.340, 0.391, 0.429] |
| 5  | 8662.48     | R9: [1.0, 1.0, 1.0, 1.0] |
| 6  | -0.1778     | R6: [0.411, 0.361, 0.770, 0.849, 0.185] |
| 7  | 2.1351      | R9: [0.097, 0.283, 0.094, 0.206, 0.354, 0.658] |
| 8  | 9.9654      | R8: [0.054, 0.274, 0.099, 0.151, 0.706, 0.443, 0.170, 0.442] |

Round 10 strategy:

- **Function 1** -- Fixed [0.5, 0.5]: confirmed zero across all rounds, no budget value
- **Function 2** -- Direct recovery to R7 best [0.715, 0.911]: NN has drifted 3 rounds in a row; gradient is unreliable -- go direct with tightest possible radius
- **Function 3** -- Return to R8 best [0.442, 0.433, 0.518]: R9 moved away and regressed by 0.008; re-anchor on confirmed peak
- **Function 4** -- Return to R4 best [0.373, 0.340, 0.391, 0.429]: 6 rounds of attempts have not beaten R4; direct recovery is the only viable option
- **Function 5** -- Fixed [1.0, 1.0, 1.0, 1.0]: at boundary maximum; every dimension is saturated -- hold to confirm ceiling
- **Function 6** -- NN gradient ascent from R6 best, radius 0.003: tightest radius yet -- landscape is extremely sharp; 0.005 caused drift in R9
- **Function 7** -- NN gradient ascent from R9 best, radius 0.01: new best confirmed, exploit it before surrogate decays
- **Function 8** -- Return to R8 best [0.054, 0.274, 0.099, 0.151, 0.706, 0.443, 0.170, 0.442]: R9 regressed; re-anchor


In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C

# -- Load initial data --------------------------------------------------------
base_path = r'C:/Users/Owner/Documents/Terraform2/capstone/Initial_data_points_starter/initial_data'
functions = {}
for i in range(1, 9):
    inputs  = np.load(f'{base_path}/function_{i}/initial_inputs.npy')
    outputs = np.load(f'{base_path}/function_{i}/initial_outputs.npy')
    functions[i] = {'inputs': inputs, 'outputs': outputs}

# -- All round results --------------------------------------------------------
round_1 = {
    1: {'input': np.array([0.781024, 0.783000]),                                                                   'output': 3.600116e-33},
    2: {'input': np.array([0.752637, 0.975656]),                                                                   'output': 0.350483},
    3: {'input': np.array([0.542581, 0.661593, 0.390176]),                                                         'output': -0.023450},
    4: {'input': np.array([0.627766, 0.478772, 0.475826, 0.299007]),                                               'output': -6.500845},
    5: {'input': np.array([0.274189, 0.896480, 0.929484, 0.928516]),                                               'output': 1961.095563},
    6: {'input': np.array([0.778186, 0.204693, 0.782552, 0.743997, 0.106401]),                                     'output': -0.639754},
    7: {'input': np.array([0.107896, 0.541672, 0.297422, 0.268118, 0.470428, 0.780970]),                           'output': 1.020486},
    8: {'input': np.array([0.106447, 0.115956, 0.072929, 0.088786, 0.453935, 0.851055, 0.538307, 0.943085]),       'output': 9.561445},
}
round_2 = {
    1: {'input': np.array([0.722566, 0.786336]),                                                                   'output': -5.837656e-24},
    2: {'input': np.array([0.338518, 0.489826]),                                                                   'output': 0.038163},
    3: {'input': np.array([0.353706, 0.737070, 0.756926]),                                                         'output': -0.103956},
    4: {'input': np.array([0.548017, 0.538375, 0.021034, 0.258912]),                                               'output': -10.973844},
    5: {'input': np.array([0.225843, 0.909370, 0.503192, 0.299769]),                                               'output': 58.713199},
    6: {'input': np.array([0.386227, 0.370287, 0.535411, 0.895749, 0.315540]),                                     'output': -0.491468},
    7: {'input': np.array([0.089709, 0.438459, 0.180432, 0.183812, 0.354751, 0.831516]),                           'output': 1.391817},
    8: {'input': np.array([0.443569, 0.599844, 0.033920, 0.013473, 0.948748, 0.067015, 0.092241, 0.036099]),       'output': 9.261666},
}
round_3 = {
    1: {'input': np.array([0.787451, 0.713163]),                                                                   'output': -5.800009e-23},
    2: {'input': np.array([0.666136, 0.924650]),                                                                   'output': 0.468232},
    3: {'input': np.array([0.432014, 0.423353, 0.507804]),                                                         'output': -0.017713},
    4: {'input': np.array([0.469321, 0.416892, 0.307101, 0.426050]),                                               'output': -1.313907},
    5: {'input': np.array([0.264682, 0.835123, 0.958851, 0.940666]),                                               'output': 1909.486564},
    6: {'input': np.array([0.263435, 0.869356, 0.592685, 0.722761, 0.544815]),                                     'output': -1.269150},
    7: {'input': np.array([0.085295, 0.293495, 0.173384, 0.126868, 0.393363, 0.660213]),                           'output': 1.709228},
    8: {'input': np.array([0.104083, 0.054438, 0.407037, 0.006596, 0.900508, 0.470715, 0.078870, 0.916641]),       'output': 9.694742},
}
round_4 = {
    1: {'input': np.array([0.001601, 0.996704]),                                                                   'output': 0.000000},
    2: {'input': np.array([0.728170, 0.898302]),                                                                   'output': 0.609560},
    3: {'input': np.array([0.505240, 0.633279, 0.561304]),                                                         'output': -0.024916},
    4: {'input': np.array([0.373386, 0.339923, 0.390620, 0.428872]),                                               'output': 0.328996},
    5: {'input': np.array([0.324189, 0.946480, 0.979484, 0.978516]),                                               'output': 3316.467309},
    6: {'input': np.array([0.286227, 0.270287, 0.435411, 0.995749, 0.215540]),                                     'output': -0.839350},
    7: {'input': np.array([0.031708, 0.132872, 0.302691, 0.030490, 0.427264, 0.704727]),                           'output': 1.243975},
    8: {'input': np.array([0.179513, 0.337380, 0.030860, 0.509674, 0.636042, 0.713639, 0.138762, 0.701544]),       'output': 9.725778},
}
round_5 = {
    1: {'input': np.array([0.250000, 0.250000]),                                                                   'output': 9.797748e-42},
    2: {'input': np.array([0.707083, 0.907229]),                                                                   'output': 0.616968},
    3: {'input': np.array([0.372278, 0.281724, 0.464231]),                                                         'output': -0.026427},
    4: {'input': np.array([0.415977, 0.352537, 0.433120, 0.452705]),                                               'output': 0.063685},
    5: {'input': np.array([0.354189, 0.976480, 1.000000, 1.000000]),                                               'output': 4191.358368},
    6: {'input': np.array([0.433578, 0.290287, 0.536384, 0.975749, 0.371492]),                                     'output': -0.693429},
    7: {'input': np.array([0.071560, 0.386831, 0.174019, 0.062513, 0.468446, 0.799137]),                           'output': 0.743942},
    8: {'input': np.array([0.074863, 0.317416, 0.147056, 0.119067, 0.672307, 0.424989, 0.136937, 0.406485]),       'output': 9.943400},
}
round_6 = {
    1: {'input': np.array([0.998394, 0.999482]),                                                                   'output': 1.699559e-191},
    2: {'input': np.array([0.944177, 0.013586]),                                                                   'output': -0.044538},
    3: {'input': np.array([0.990867, 0.502049, 0.536263]),                                                         'output': -0.018409},
    4: {'input': np.array([0.385593, 0.396692, 0.459001, 0.481881]),                                               'output': -2.165793},
    5: {'input': np.array([0.404189, 1.000000, 1.000000, 1.000000]),                                               'output': 4544.476796},
    6: {'input': np.array([0.411313, 0.361437, 0.769565, 0.849107, 0.184824]),                                     'output': -0.177831},
    7: {'input': np.array([0.099423, 0.293146, 0.092186, 0.195886, 0.363972, 0.648441]),                           'output': 2.037637},
    8: {'input': np.array([0.025950, 0.163720, 0.116687, 0.391130, 0.980564, 0.689047, 0.147088, 0.049991]),       'output': 9.842279},
}
round_7 = {
    1: {'input': np.array([0.500000, 0.500000]),                                                                   'output': 2.675288e-09},
    2: {'input': np.array([0.715128, 0.911455]),                                                                   'output': 0.668564},
    3: {'input': np.array([0.462014, 0.393353, 0.537804]),                                                         'output': -0.033122},
    4: {'input': np.array([0.368239, 0.359923, 0.404251, 0.448872]),                                               'output': 0.137239},
    5: {'input': np.array([0.554189, 1.000000, 1.000000, 1.000000]),                                               'output': 4797.241552},
    6: {'input': np.array([0.394491, 0.365429, 0.829565, 0.909107, 0.124826]),                                     'output': -0.358740},
    7: {'input': np.array([0.049423, 0.243146, 0.042186, 0.145886, 0.313972, 0.676699]),                           'output': 1.815362},
    8: {'input': np.array([0.024863, 0.267416, 0.097056, 0.069067, 0.722307, 0.474989, 0.086937, 0.356485]),       'output': 9.929976},
}
round_8 = {
    1: {'input': np.array([0.950000, 0.050000]),                                                                   'output': 4.406064e-291},
    2: {'input': np.array([0.711286, 0.881455]),                                                                   'output': 0.527492},
    3: {'input': np.array([0.442014, 0.433337, 0.517804]),                                                         'output': -0.012923},
    4: {'input': np.array([0.373386, 0.339923, 0.390620, 0.436852]),                                               'output': 0.211078},
    5: {'input': np.array([0.900000, 1.000000, 1.000000, 1.000000]),                                               'output': 7073.445506},
    6: {'input': np.array([0.408135, 0.337445, 0.729565, 0.809107, 0.224822]),                                     'output': -0.298438},
    7: {'input': np.array([0.089423, 0.283146, 0.072186, 0.185421, 0.380716, 0.628441]),                           'output': 1.870841},
    8: {'input': np.array([0.053794, 0.273991, 0.098763, 0.151316, 0.705994, 0.442635, 0.169673, 0.441635]),       'output': 9.965371},
}
round_9 = {
    1: {'input': np.array([0.500000, 0.500000]),                                                                   'output': 2.675288e-09},
    2: {'input': np.array([0.723005, 0.921455]),                                                                   'output': 0.434550},
    3: {'input': np.array([0.432014, 0.443337, 0.507804]),                                                         'output': -0.020493},
    4: {'input': np.array([0.383386, 0.338835, 0.380620, 0.418872]),                                               'output': 0.284886},
    5: {'input': np.array([1.000000, 1.000000, 1.000000, 1.000000]),                                               'output': 8662.482500},
    6: {'input': np.array([0.416313, 0.366437, 0.764565, 0.845316, 0.183661]),                                     'output': -0.304636},
    7: {'input': np.array([0.096960, 0.283146, 0.093725, 0.205886, 0.353972, 0.658441]),                           'output': 2.135136},
    8: {'input': np.array([0.024580, 0.283803, 0.079445, 0.178980, 0.684914, 0.437512, 0.144794, 0.471447]),       'output': 9.943938},
}

# -- GP-UCB local search ------------------------------------------------------
def gp_ucb_local(X, y, center, radius, n_candidates=3000, beta=2.0):
    center = np.array(center)
    lo = np.clip(center - radius, 0.0, 1.0)
    hi = np.clip(center + radius, 0.0, 1.0)
    np.random.seed(42)
    candidates = np.random.uniform(lo, hi, (n_candidates, len(center)))

    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_norm = (y - y_mean) / y_std

    kernels = [
        C(1.0) * Matern(length_scale=0.05, nu=2.5),
        C(1.0) * Matern(length_scale=0.10, nu=1.5),
    ]
    ensemble_ucb = np.zeros(n_candidates)
    for kernel in kernels:
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=2, normalize_y=False)
        gp.fit(X, y_norm)
        mu, sigma = gp.predict(candidates, return_std=True)
        ensemble_ucb += mu + beta * sigma
    return candidates[np.argmax(ensemble_ucb)]

# -- Per-function config -------------------------------------------------------
configs = {
    2: {'center': [0.715128, 0.911455], 'radius': 0.005},
    3: {'center': [0.442014, 0.433337, 0.517804], 'radius': 0.005},
    4: {'center': [0.373386, 0.339923, 0.390620, 0.428872], 'radius': 0.005},
    6: {'center': [0.411313, 0.361437, 0.769565, 0.849107, 0.184824], 'radius': 0.003},
    7: {'center': [0.096960, 0.283146, 0.093725, 0.205886, 0.353972, 0.658441], 'radius': 0.010},
    8: {'center': [0.053794, 0.273991, 0.098763, 0.151316, 0.705994, 0.442635, 0.169673, 0.441635], 'radius': 0.005},
}

print('All-time bests after R9:')
bests = {2: 0.668564, 3: -0.012923, 4: 0.328996, 5: 8662.4825, 6: -0.177831, 7: 2.135136, 8: 9.965371}
for i in range(1, 9):
    print(f'  F{i}: {bests.get(i, 0.0):.6g}')
print()

for i in range(1, 9):
    X = np.vstack([
        functions[i]['inputs'],
        round_1[i]['input'], round_2[i]['input'], round_3[i]['input'],
        round_4[i]['input'], round_5[i]['input'], round_6[i]['input'],
        round_7[i]['input'], round_8[i]['input'], round_9[i]['input'],
    ])
    y = np.concatenate([
        functions[i]['outputs'],
        [round_1[i]['output']], [round_2[i]['output']], [round_3[i]['output']],
        [round_4[i]['output']], [round_5[i]['output']], [round_6[i]['output']],
        [round_7[i]['output']], [round_8[i]['output']], [round_9[i]['output']],
    ])

    if i == 1:
        query  = np.array([0.5, 0.5])
        method = 'fixed -- F1 confirmed zero'
    elif i == 5:
        query  = np.array([1.0, 1.0, 1.0, 1.0])
        method = 'fixed -- F5 boundary maximum'
    else:
        cfg   = configs[i]
        query = gp_ucb_local(X, y, cfg['center'], cfg['radius'])
        method = f'GP-UCB local radius={cfg["radius"]} around {cfg["center"]}'

    print(f'--- Function {i} ---')
    print(f'Method:    {method}')
    print(f'Query:     {query}')
    print(f'Formatted: {"-".join(f"{v:.6f}" for v in query)}')
    print()


## Round 11 -- Exploit R10 New Bests + F6 Ultra-Tight Probe

R10 results: **3 new all-time bests** -- F4, F7, F8 all improved. F5 ceiling reconfirmed.

| Fn | R9 | R10 | All-time best | Round |
|----|-----|-----|--------------|-------|
| F1 | 2.68e-9 | 2.68e-9 | ~0 | RETIRED |
| F2 | 0.4346 | 0.5044 | **0.6686** | R7 |
| F3 | -0.0205 | -0.0233 | **-0.0129** | R8 |
| F4 | 0.2849 | **0.4394** | **0.4394** | R10 NEW |
| F5 | 8662.48 | 8662.48 | 8662.48 | R9 ceiling |
| F6 | -0.3046 | -0.3022 | **-0.1778** | R6 |
| F7 | 2.1351 | **2.2164** | **2.2164** | R10 NEW |
| F8 | 9.9439 | **9.9663** | **9.9663** | R10 NEW |

### R11 Strategy
- **F1**: Fixed [0.5, 0.5] -- confirmed zero
- **F2**: GP-UCB +/-0.005 anchored on R7 best -- recovering toward 0.6686
- **F3**: GP-UCB +/-0.005 anchored on R8 best -- recovering -0.0129
- **F4**: GP-UCB +/-0.005 anchored on R10 new best -- active improvement seam
- **F5**: Fixed [1,1,1,1] -- boundary ceiling confirmed
- **F6**: Dense random probe +/-0.0001 around R6 best -- GP not useful at this scale; R10 was 0.003 away in every dimension and still returned -0.302 vs -0.178; ultra-tight search to probe the immediate spike neighbourhood
- **F7**: GP-UCB +/-0.010 anchored on R10 new best -- continuing upward trend
- **F8**: GP-UCB +/-0.005 anchored on R10 new best -- marginal gains, tight exploit

In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C

np.random.seed(42)

# Training data rounds 1-10
r = {
    2: dict(
        X=[[0.752637,0.975656],[0.338518,0.489826],[0.666136,0.92465],
           [0.72817,0.898302],[0.707083,0.907229],[0.944177,0.013586],
           [0.715128,0.911455],[0.711286,0.881455],[0.723005,0.921455],
           [0.71834,0.906471]],
        y=[0.35048,0.03816,0.46823,0.60956,0.61697,-0.04454,
           0.66856,0.52749,0.43455,0.50438]),
    3: dict(
        X=[[0.542581,0.661593,0.390176],[0.353706,0.73707,0.756926],
           [0.432014,0.423353,0.507804],[0.50524,0.633279,0.561304],
           [0.372278,0.281724,0.464231],[0.990867,0.502049,0.536263],
           [0.462014,0.393353,0.537804],[0.442014,0.433337,0.517804],
           [0.432014,0.443337,0.507804],[0.446819,0.433275,0.512901]],
        y=[-0.02345,-0.10396,-0.01771,-0.02492,-0.02643,-0.01841,
           -0.03312,-0.01292,-0.02049,-0.02334]),
    4: dict(
        X=[[0.627766,0.478772,0.475826,0.299007],
           [0.548017,0.538375,0.021034,0.258912],
           [0.469321,0.416892,0.307101,0.42605],
           [0.373386,0.339923,0.39062,0.428872],
           [0.415977,0.352537,0.43312,0.452705],
           [0.385593,0.396692,0.459001,0.481881],
           [0.368239,0.359923,0.404251,0.448872],
           [0.373386,0.339923,0.39062,0.436852],
           [0.383386,0.338835,0.38062,0.418872],
           [0.37675,0.344667,0.395376,0.424525]],
        y=[-6.50085,-10.97384,-1.31391,0.32900,0.06368,-2.16579,
           0.13724,0.21108,0.28489,0.43936]),
    7: dict(
        X=[[0.107896,0.541672,0.297422,0.268118,0.470428,0.78097],
           [0.089709,0.438459,0.180432,0.183812,0.354751,0.831516],
           [0.085295,0.293495,0.173384,0.126868,0.393363,0.660213],
           [0.031708,0.132872,0.302691,0.03049,0.427264,0.704727],
           [0.07156,0.386831,0.174019,0.062513,0.468446,0.799137],
           [0.099423,0.293146,0.092186,0.195886,0.363972,0.648441],
           [0.049423,0.243146,0.042186,0.145886,0.313972,0.676699],
           [0.089423,0.283146,0.072186,0.185421,0.380716,0.628441],
           [0.09696,0.283146,0.093725,0.205886,0.353972,0.658441],
           [0.087791,0.276113,0.103458,0.215188,0.344071,0.667477]],
        y=[1.02049,1.39182,1.70923,1.24398,0.74394,2.03764,
           1.81536,1.87084,2.13514,2.21642]),
    8: dict(
        X=[[0.106447,0.115956,0.072929,0.088786,0.453935,0.851055,0.538307,0.943085],
           [0.443569,0.599844,0.03392,0.013473,0.948748,0.067015,0.092241,0.036099],
           [0.104083,0.054438,0.407037,0.006596,0.900508,0.470715,0.07887,0.916641],
           [0.179513,0.33738,0.03086,0.509674,0.636042,0.713639,0.138762,0.701544],
           [0.074863,0.317416,0.147056,0.119067,0.672307,0.424989,0.136937,0.406485],
           [0.02595,0.16372,0.116687,0.39113,0.980564,0.689047,0.147088,0.049991],
           [0.024863,0.267416,0.097056,0.069067,0.722307,0.474989,0.086937,0.356485],
           [0.053794,0.273991,0.098763,0.151316,0.705994,0.442635,0.169673,0.441635],
           [0.02458,0.283803,0.079445,0.17898,0.684914,0.437512,0.144794,0.471447],
           [0.058054,0.269229,0.094279,0.155903,0.703766,0.446369,0.168561,0.437826]],
        y=[9.56145,9.26167,9.69474,9.72578,9.94340,9.84228,
           9.92998,9.96537,9.94394,9.96629])
}

def gp_ucb(X_train, y_train, center, radius, n=5000, beta=2.0):
    X_train, y_train, center = np.array(X_train), np.array(y_train), np.array(center)
    cands = np.clip(center + np.random.uniform(-radius, radius, (n, len(center))), 0, 1)
    kernel = C(1.0)*Matern(length_scale=0.05, nu=2.5) + C(1.0)*Matern(length_scale=0.10, nu=2.5)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=True)
    gp.fit(X_train, y_train)
    mu, sigma = gp.predict(cands, return_std=True)
    return cands[np.argmax(mu + beta * sigma)]

queries = {}

# F1: confirmed zero
queries[1] = np.array([0.5, 0.5])

# F2: recover R7 best, GP-UCB +-0.005
queries[2] = gp_ucb(r[2]["X"], r[2]["y"], [0.715128, 0.911455], 0.005)

# F3: recover R8 best, GP-UCB +-0.005
queries[3] = gp_ucb(r[3]["X"], r[3]["y"], [0.442014, 0.433337, 0.517804], 0.005)

# F4: exploit R10 new best, GP-UCB +-0.005
queries[4] = gp_ucb(r[4]["X"], r[4]["y"], [0.37675, 0.344667, 0.395376, 0.424525], 0.005)

# F5: boundary ceiling confirmed
queries[5] = np.array([1.0, 1.0, 1.0, 1.0])

# F6: ultra-tight +-0.0001 random probe around R6 best -- GP not useful at this scale
f6_center = np.array([0.411313, 0.361437, 0.769565, 0.849107, 0.184824])
f6_cands = np.clip(f6_center + np.random.uniform(-0.0001, 0.0001, (5000, 5)), 0, 1)
queries[6] = f6_cands[np.random.randint(5000)]

# F7: exploit R10 new best, GP-UCB +-0.010
queries[7] = gp_ucb(r[7]["X"], r[7]["y"],
                    [0.087791, 0.276113, 0.103458, 0.215188, 0.344071, 0.667477], 0.010)

# F8: exploit R10 new best, GP-UCB +-0.005
queries[8] = gp_ucb(r[8]["X"], r[8]["y"],
                    [0.058054, 0.269229, 0.094279, 0.155903, 0.703766,
                     0.446369, 0.168561, 0.437826], 0.005)

print("=" * 60)
print("R11 QUERIES -- submit to portal")
print("=" * 60)
for fn, q in queries.items():
    print(f"F{fn}: {chr(45).join(f'{x:.6f}' for x in q)}")